In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import glob, sys, os

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

SESSION_DIR     = REPO_ROOT / 'notebooks' / 'test_session'
CONFIG_PATH     = REPO_ROOT / 'notebooks' / 'config.toml'
INTRINSICS_FILE = Path('/storage/project/r-jmarkowitz30-0/shared/active_lab_members/markowitz_jeffrey/active_projects/mouse_open_field_lucid_rig_da_photometry/intrinsics_lucid_rig.toml')
SKELETON_JSON   = REPO_ROOT / 'notebooks' / 'skeleton.json'
TRANSFORMS_PATH = REPO_ROOT / 'notebooks' / 'avg_transforms.toml'

CENTROID = "/storage/home/hcoda1/3/triesenmy3/lab_folder/sleap_nn_models/models/centroid_unet"
CI = '/storage/home/hcoda1/3/triesenmy3/lab_folder/sleap_nn_models/models/convnext-large_seed-4'

REFERENCE_CAMERA = 'Lucid Vision Labs-HTP003S-001-224500508'
VERSION_NUM      = 1
CABLE            = False
CONDA_ENV_NAME   = 'sleap-nn-env'

NODE_NAMES = [
    'tail_tip', 'tail_middle', 'tail_base', 'back_bottom',
    'back_middle_lower', 'back_middle_upper', 'back_top',
    'left_ear', 'right_ear', 'snout',
    'left_hip', 'right_hip', 'left_shoulder', 'right_shoulder',
]

video_paths = sorted(glob.glob(str(SESSION_DIR / '_proc' / '*.avi')))

INFERENCE_OUTPUT_PATH = SESSION_DIR / '_proc' / f'_keypoints_v{VERSION_NUM}'
KEYPOINTS_OUTPUT_PATH = SESSION_DIR / '_proc' / f'_kpoints_v{VERSION_NUM}_3d'
RENDERS_OUTPUT_PATH   = SESSION_DIR / '_proc' / 'renders'

RENDERS_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f'Session: {SESSION_DIR}')
print(f'Videos found: {len(video_paths)}')

In [ ]:
from depth_keys.experiment.trial import Trial

In [ ]:
trial = Trial(
    trial_id=SESSION_DIR.name,
    video_paths=video_paths,
    version_num=VERSION_NUM,
    base_dir=str(SESSION_DIR.parent),
    node_names=NODE_NAMES,
    video_extension='.avi',
    inference_output_path=str(INFERENCE_OUTPUT_PATH),
    keypoints_output_path=str(KEYPOINTS_OUTPUT_PATH),
    reference_camera=REFERENCE_CAMERA,
    intrinsics_file=str(INTRINSICS_FILE),
    cable=CABLE,
    conda_env_name=CONDA_ENV_NAME,
    transforms_path=TRANSFORMS_PATH,
)

In [ ]:
# process_session: 2D keypoint prediction
trial.predict_keypoints(ci_model_path=CI, centroid_model_path=CENTROID)

In [ ]:
# post_process: 2D -> 3D conversion
trial.compute_3d_keypoints(config_path=str(CONFIG_PATH))

In [ ]:
# visualize: render keypoint overlay + 3D matplotlib video
alt_key_path = os.path.join(trial.keypoints_output_path, 'merged_keypoints.h5')

trial.visualize(
    matplot_viz=True,
    overlay_viz=True,
    output_dir=str(RENDERS_OUTPUT_PATH),
    skeleton_json_path=str(SKELETON_JSON),
    alt_key_path=alt_key_path,
)

In [ ]:
for p in [INFERENCE_OUTPUT_PATH, KEYPOINTS_OUTPUT_PATH, RENDERS_OUTPUT_PATH]:
    print(f'{p.name} -> {"exists" if p.exists() else "missing"}')